# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json)

This dataset includes survey data on mental health indicators, demographics, and standardized score assessments (GAD-7, PHQ-9, PSQ) for residents of Kilifi County, Kenya.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Accessing metadata
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Keywords: {', '.join(dataset.metadata.keywords)}")

## 2. Data Overview
Review available record sets, fields (variables), and their `@id`s.

**Note:** All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets and their @id
record_sets = dataset.record_sets
print("Available Record Sets:")
for rs in record_sets:
    print(f"  - {rs['@id']} (name: {rs['name']})")

# For this dataset, let's print fields for the main records set
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    fields = dataset.fields(record_set=main_record_set_id)
    print(f"\nFields for Record Set {main_record_set_id}:")
    for field in fields:
        print(f"  - {field['@id']} (name: {field['name']}, dataType: {field.get('dataType')})")
else:
    print("No record sets found in dataset.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# For the FAIR² dataset, typically only one main record set is present.
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()

# Let's use the first record set for demonstration
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and not dataframes[main_record_set_id].empty:
    print(f"Columns for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()
else:
    print("No records loaded for main record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include removing outliers, transforming data distributions, and grouping data by key attributes.

All column references are by their `@id` (use the actual `@id` as seen in step 2 and 3).

In [ ]:
# Example EDA on GAD-7 score field
# First, select columns by their @id from the schema overview
# NOTE: Replace the field @id values below with those discovered in step 2 if necessary.
# For demonstration, we use plausible @id values:
gad7_score_id = 'https://api.app.sen.science/frontiers/7160411/field/gad7_score'  # Change to real @id
education_level_id = 'https://api.app.sen.science/frontiers/7160411/field/level_of_education'  # Change to real @id

df = dataframes[main_record_set_id]

if gad7_score_id in df.columns:
    # Filter for GAD-7 scores > 10 (moderate/severe anxiety)
    threshold = 10
    filtered_df = df[df[gad7_score_id] > threshold]
    print(f"Filtered records with GAD-7 score (@id {gad7_score_id}) > {threshold}:")
    print(filtered_df.head())

    # Normalize GAD-7 scores
    filtered_df[f"{gad7_score_id}_normalized"] = (filtered_df[gad7_score_id] - filtered_df[gad7_score_id].mean()) / filtered_df[gad7_score_id].std()
    print(f"Normalized GAD-7 scores for filtered records:")
    print(filtered_df[[gad7_score_id, f"{gad7_score_id}_normalized"].head()])

    # Group by education level if that field exists
    group_field_id = education_level_id
    if group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[gad7_score_id].mean().reset_index()
        print(f"Average GAD-7 score grouped by Level of Education (@id {group_field_id}):")
        print(grouped_df.head())
else:
    print(f"Field {gad7_score_id} not found in DataFrame.")

## 5. Visualization
Visualize distributions of GAD-7 scores and their relationship with level of education, referencing columns by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize GAD-7 score distribution
if gad7_score_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[gad7_score_id].dropna(), bins=10, kde=True)
    plt.title('Distribution of GAD-7 Scores')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Count')
    plt.show()

    # Compare GAD-7 scores across education levels
    if education_level_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[education_level_id], y=df[gad7_score_id])
        plt.title('GAD-7 Score by Level of Education')
        plt.xlabel('Level of Education (@id)')
        plt.ylabel('GAD-7 Score')
        plt.xticks(rotation=30)
        plt.show()

## 6. Conclusion
This notebook explored the FAIR² Mental Health Survey dataset for Kilifi County, Kenya using the `mlcroissant` library.
- We loaded the dataset and reviewed available record sets and fields, referencing each entity by its unique `@id`.
- Data manipulation leveraged field `@id` for reliable, schema-compliant access and processing.
- Exploratory analysis and visualizations provided initial insights on GAD-7 scores and their relationship to education level.

Further analysis can extend to other variables such as PHQ-9 and PSQ scores, and to demographic breakdowns to better understand mental health trends in the county.